In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/11011
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  2035.54957768875
RUN  2 , total integrated cost =  1224.3485002247332
RUN  3 , total integrated cost =  793.4673040184703
RUN  4 , total integrated cost =  661.6465991628415
RUN  5 , total integrated cost =  563.5530673030038
RUN  6 , total integrated cost =  511.7921633665966
RUN  7 , total integrated cost =  472.099200643472
RUN  8 , total integrated cost =  448.29888039990584
RUN  9 , total integrated cost =  433.6686661602265
RUN  10 , total integrated cost =  423.5313396857817
RUN  11 , total integrated cost =  416.33940617674216
RUN  12 , total integrated cost =  410.1264783536225
RUN  13 , total integrated cost =  404.98711614435166
RUN  14 , total integrated cost =  400.4359002887344
RUN  15 , total integrated cost =  396.5913784138585
RUN  16 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  581 , total integrated cost =  279.40504987585007
Improved over  581  iterations in  79.21156634390354  seconds by  94.51855673714886  percent.
Problem in initial value trasfer:  Vmean_exc -67.58627763104539 -67.5935861033221
weight =  182.43370441817842
set cost params:  1.0 182.43370441817842 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5088.731108792851
Gradient descend method:  None
RUN  1 , total integrated cost =  5069.659347871975
RUN  2 , total integrated cost =  5069.605924105929
RUN  3 , total integrated cost =  5069.604828700057
RUN  4 , total integrated cost =  5069.604809128109
RUN  5 , total integrated cost =  5069.60480617012
RUN  6 , total integrated cost =  5069.604806097991
RUN  7 , total integrated cost =  5069.604806097981
RUN  8 , total integrated cost =  5069.604806097977
RUN  9 , total integrated cost =  5069.6048060979765
RUN  10 , total integrated cost =  5069.60480609796
RUN  11 , total integ

ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  5069.604806097952
RUN  13 , total integrated cost =  5069.604806097952
Control only changes marginally.
RUN  13 , total integrated cost =  5069.604806097952
Improved over  13  iterations in  1.7555021792650223  seconds by  0.3758560294500626  percent.
Problem in initial value trasfer:  Vmean_exc -66.30942524981849 -66.3392365181478
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  10803.948671292466
RUN  2 , total integrated cost =  10757.43170651875
RUN  3 , total integrated cost =  10755.407271136808
RUN  4 , total integrated cost =  10755.347271393324
RUN  5 , total integrated cost =  10755.337199833131
RUN  6 , total integrated cost =  10755.33709502086
RUN  7 , total integrated cost =  10755.337094515897
RUN  8 , total integrated cost =  10755.337094515891
RUN  9

...

RUN  1 , total integrated cost =  22819.982851368484
Control only changes marginally.
RUN  1 , total integrated cost =  22819.982851368484
Improved over  1  iterations in  0.20086591131985188  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.699724647788116 -56.699786224706905
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  32545.10794721557
RUN  2 , total integrated cost =  32538.30962933377
RUN  3 , total integrated cost =  32538.055908753646
RUN  4 , total integrated cost =  32537.758472028298
RUN  5 , total integrated cost =  32537.599407041733
RUN  6 , total integrated cost =  32537.494088188734
RUN  7 , total integrated cost =  32537.37774804351
RUN  8 , total integrated cost =  32537.314586068354
RUN  9 , total integrated cost =  32537.234813033534
RUN  10 , total int

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  39 , total integrated cost =  32536.394155451704
Improved over  39  iterations in  2.6756983026862144  seconds by  2.263911523765202  percent.
Problem in initial value trasfer:  Vmean_exc -56.703764416419084 -56.70375172829622
weight =  10.23163516762958
set cost params:  1.0 10.23163516762958 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32543.0565165371
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32543.0565165371
Control only changes marginally.
RUN  1 , total integrated cost =  32543.0565165371
Improved over  1  iterations in  0.20415204763412476  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.703764416419084 -56.70375172829622


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
[0, 5] []
closest index  5
set 

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  6603.555449609012
Control only changes marginally.
RUN  14 , total integrated cost =  6603.555449609012
Improved over  14  iterations in  1.9450742918998003  seconds by  29.545603424603755  percent.
Problem in initial value trasfer:  Vmean_exc -56.626461631645654 -56.626733978788636
weight =  13.797804167375286
set cost params:  1.0 13.797804167375286 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6946.645562660216
Gradient descend method:  None
RUN  1 , total integrated cost =  6897.555496559536
RUN  2 , total integrated cost =  6896.489526359109
RUN  3 , total integrated cost =  6896.451415844507
RUN  4 , total integrated cost =  6896.450719018574
RUN  5 , total integrated cost =  6896.450715353966
RUN  6 , total integrated cost =  6896.450715338921
RUN  7 , total integrated cost =  6896.45071533791
RUN  8 , total integrated cost =  6896.450715337868
RUN  9 , total integrated cost =  6896.450715337867


ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  6896.450715337867
Control only changes marginally.
RUN  10 , total integrated cost =  6896.450715337867
Improved over  10  iterations in  1.356311321258545  seconds by  0.7225767727686616  percent.
Problem in initial value trasfer:  Vmean_exc -56.62815709755465 -56.6284441501808
-------  15 0.4500000000000001 0.4500000000000002
[0, 5] []
closest index  5
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13285.219985545464
Gradient descend method:  None
RUN  1 , total integrated cost =  10949.147331890894
RUN  2 , total integrated cost =  10768.484597260176
RUN  3 , total integrated cost =  10756.120078308004
RUN  4 , total integrated cost =  10755.596385308492
RUN  5 , total integrated cost =  10755.456348602871
RUN  6 , total integrated cost =  10755.431261852586
RUN  7 , total integrated cost =  10755.413778746743
RUN  8 , total integrated cost =  10755.408998964269
RUN  9 , 

...

RUN  8 , total integrated cost =  9483.232059792264
RUN  9 , total integrated cost =  9483.232059792264
Control only changes marginally.
RUN  9 , total integrated cost =  9483.232059792264
Improved over  9  iterations in  0.9135622046887875  seconds by  0.0023219856412168838  percent.
Problem in initial value trasfer:  Vmean_exc -56.64696788666435 -56.647089189754944
-------  145 0.5750000000000002 0.9000000000000006
[0, 5] [5]
closest index  0
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33808.54226593651
Gradient descend method:  None
RUN  1 , total integrated cost =  32713.705639729145
RUN  2 , total integrated cost =  32682.819809821678
RUN  3 , total integrated cost =  32600.362514360164
RUN  4 , total integrated cost =  32594.06049441509
RUN  5 , total integrated cost =  32587.901684962882
RUN  6 , total integrated cost =  32585.522121943937
RUN  7 , total integrated cost =  32582.664208347847
RUN  8 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  52 , total integrated cost =  32544.160887071997
Improved over  52  iterations in  5.037222763523459  seconds by  3.739828144375295  percent.
Problem in initial value trasfer:  Vmean_exc -56.70379363119275 -56.703784981867756
weight =  10.22919336663617
set cost params:  1.0 10.22919336663617 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32551.482381204627
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32551.482381204627
Control only changes marginally.
RUN  1 , total integrated cost =  32551.482381204627
Improved over  1  iterations in  0.19646973349153996  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70379363119275 -56.703784981867756
------------------------------------------------------------
-------------------- 2
------------------------------------------------------------
found solution:  [0, 5]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
closest index  -1
set cost params:  1.0 10.0 0.0
all options tried already
-------  15 0.4500000000000001 0.4500000000000002
closest index  -1
set cost params:  1.0 10.0 0.0
all options tried already
-------  20 0.4500000000000001 0.4750000000000002
closest index  -1
set cost params:  1.0 10.0 0.0
all options tried already
-------  25 0.4250000000000001 0.500

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  182.429971648486
set cost params:  1.0 182.429971648486 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5069.501293441041
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5069.501293441041
Control only changes marginally.
RUN  1 , total integrated cost =  5069.501293441041
Improved over  1  iterations in  0.3488035500049591  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.30942524981849 -66.3392365181478
-------  10 0.4250000000000001 0.42500000000000016
no convergence
weight =  17.229455716134428
set cost params:  1.0 17.229455716134428 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7113.503510608209
Gradient descend method:  None
RUN  1 , total integrated cost =  7091.983396512341
RUN  2 , total integrated cost =  7091.571190004675
RUN  3 , total integrated cost =  7091.551688070922
RUN  4 , total integrated cost =  7091.551653161882
RUN  5 , total integrated cost =  7091.551653161877
RUN  6 , total integrated cost =  7091.551653161873


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  7091.551653161873
Control only changes marginally.
RUN  7 , total integrated cost =  7091.551653161873
Improved over  7  iterations in  1.1135559678077698  seconds by  0.308594174636994  percent.
Problem in initial value trasfer:  Vmean_exc -56.629433542911585 -56.629702710307996
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  13.443429479328472
set cost params:  1.0 13.443429479328472 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10997.446778552026
Gradient descend method:  None
RUN  1 , total integrated cost =  10993.378535644302
RUN  2 , total integrated cost =  10993.327791972235
RUN  3 , total integrated cost =  10993.315374330195
RUN  4 , total integrated cost =  10993.314369567663
RUN  5 , total integrated cost =  10993.312819332556
RUN  6 , total integrated cost =  10993.312801933418
RUN  7 , total integrated cost =  10993.312801933414
RUN  8 , total integrated cost =  10993.31280193341


...

RUN  11 , total integrated cost =  27561.235881858887
RUN  12 , total integrated cost =  27561.235881858887
Control only changes marginally.
RUN  12 , total integrated cost =  27561.235881858887
Improved over  12  iterations in  1.154968723654747  seconds by  0.00012615908671875786  percent.
Problem in initial value trasfer:  Vmean_exc -56.70373983838414 -56.70377285990558
-------  125 0.47500000000000014 0.8500000000000005
no convergence
weight =  7.859841672759746
set cost params:  1.0 7.859841672759746 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13784.467014229022
Gradient descend method:  None
RUN  1 , total integrated cost =  13784.459043939007
RUN  2 , total integrated cost =  13784.448335901374
RUN  3 , total integrated cost =  13784.440578612152
RUN  4 , total integrated cost =  13784.421875414797
RUN  5 , total integrated cost =  13784.40911679329
RUN  6 , total integrated cost =  13784.392858673891
RUN  7 , total integrated cost =  13784.387825

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  57 , total integrated cost =  13784.01294823539
Improved over  57  iterations in  6.67974753677845  seconds by  0.003294040989501923  percent.
Problem in initial value trasfer:  Vmean_exc -56.673387283311264 -56.67353550487379
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
no convergence
weight =  8.49297589829742
set cost params:  1.0 8.49297589829742 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9412.365230435662
Gradient descend method:  None
RUN  1 , total integrated cost =  9412.357994463558
RUN  2 , total integrated cost =  9412.353619605794
RUN  3 , total integrated cost =  9412.342056333426
RUN  4 , total integrated cost =  9412.337070951919
RUN  5 , total integrated cost =  9412.318880576164
RUN  6 , total integrated cost =  9412.309842813562
RUN  7 , total integrated 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  109 , total integrated cost =  9411.681234757903
Improved over  109  iterations in  11.916667278856039  seconds by  0.00726699040053802  percent.
Problem in initial value trasfer:  Vmean_exc -56.64665639871473 -56.64678572636375
-------  145 0.5750000000000002 0.9000000000000006
converged for  145


In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [False, False], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  85.04637483932316
Gradient descend method:  None
RUN  1 , total integrated cost =  74.50755124290681
RUN  2 , total integrated cost =  13.935492260542444
RUN  3 , total integrated cost =  6.0253407390812725
RUN  4 , total integrated cost =  5.844293900970386
RUN  5 , total integrated cost =  5.843365653604839
RUN  6 , total integrated cost =  5.8432721189433705
RUN  7 , total integrated cost =  5.843193510390023
RUN  8 , total integrated cost =  5.84317935665636
RUN  9 , total integrated cost =  5.84312784773981
RUN  10 , total integrated cost =  5.843092211392456
RUN  11 , total integrated cost =  5.8429116645949595
RUN  12 , total integrated cost =  5.842717847618612
RUN  13 , total integrated cost =  5.842687967107069
RUN  14 , total integrated cost =  5.842641958805371
RUN  15 , total integrated cost =  5.842628827419412
RUN  16 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  111 , total integrated cost =  5.817279752913098
Improved over  111  iterations in  37.136746941134334  seconds by  93.15987334686093  percent.
Problem in initial value trasfer:  Vmean_exc -67.93149694916386 -67.93431256907165
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  16.90868289111125
Gradient descend method:  HS
RUN  1 , total integrated cost =  16.76730465561933
RUN  2 , total integrated cost =  16.748086563690148
RUN  3 , total integrated cost =  16.745525070051023
RUN  4 , total integrated cost =  16.74551144797777
RUN  5 , total integrated cost =  16.743400812185055
RUN  6 , total integrated cost =  16.739466111307348
RUN  7 , total integrated cost =  16.736956607971
RUN  8 , total integrated cost =  16.73582830533639
RUN  9 , total integrated cost =  16.734441837509415
RUN  10 , total integrated cost =  16.73434714190043
RUN  11 , total integrated cost =  16.73276653

ERROR:root:Problem in initial value trasfer


RUN  1000 , total integrated cost =  16.566854888650823
RUN  1000 , total integrated cost =  16.566854888650823
Improved over  1000  iterations in  277.22852182388306  seconds by  2.021612237107618  percent.
Problem in initial value trasfer:  Vmean_exc -67.95122689310863 -67.95393555333855
weight =  30766.999493323492
set cost params:  1.0 30766.999493323492 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.651715820824
Gradient descend method:  None
RUN  1 , total integrated cost =  5095.048295718784
RUN  2 , total integrated cost =  5095.045648737772
RUN  3 , total integrated cost =  5095.045325971771
RUN  4 , total integrated cost =  5095.0450340787975
RUN  5 , total integrated cost =  5095.045033647276
RUN  6 , total integrated cost =  5095.044726773434
RUN  7 , total integrated cost =  5095.044413741482
RUN  8 , total integrated cost =  5095.04409532555
RUN  9 , total integrated cost =  5095.043806405825
RUN  10 , total integrated cost =  5095.043806

...

Control only changes marginally.
RUN  249 , total integrated cost =  52.772912370716156
Improved over  249  iterations in  64.07506407238543  seconds by  99.8366398652252  percent.
Problem in initial value trasfer:  Vmean_exc -64.98773439473467 -65.00342392383172
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1364.9895768486563
Gradient descend method:  HS
RUN  1 , total integrated cost =  1307.8160420886702
RUN  2 , total integrated cost =  1305.787114912977
RUN  3 , total integrated cost =  1305.0619494737919
RUN  4 , total integrated cost =  1305.0593926510057
RUN  5 , total integrated cost =  1305.0341536232374
RUN  6 , total integrated cost =  1305.03261018948
RUN  7 , total integrated cost =  1305.0324544444136
RUN  8 , total integrated cost =  1305.0074144751432
RUN  9 , total integrated cost =  1305.0074144751418
RUN  10 , total integrated cost =  1305.0074144751416


ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  1305.0074144751416
Control only changes marginally.
RUN  11 , total integrated cost =  1305.0074144751416
Improved over  11  iterations in  2.6954364329576492  seconds by  4.394331164930591  percent.
Problem in initial value trasfer:  Vmean_exc -60.52981379140472 -60.533234304748966
weight =  2549.9473047911056
set cost params:  1.0 2549.9473047911056 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32906.92676215423
Gradient descend method:  None
RUN  1 , total integrated cost =  32391.935646172893
RUN  2 , total integrated cost =  32391.01284111707
RUN  3 , total integrated cost =  32386.994799118966
RUN  4 , total integrated cost =  32384.970130980324
RUN  5 , total integrated cost =  32384.952975441767
RUN  6 , total integrated cost =  32384.558768198636
RUN  7 , total integrated cost =  32384.2006269347
RUN  8 , total integrated cost =  32384.18637043356
RUN  9 , total integrated cost =  32384.1420305213
RUN  10 , total

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  669 , total integrated cost =  21546.95801815413
Improved over  669  iterations in  99.5849388986826  seconds by  34.52151222175215  percent.
Problem in initial value trasfer:  Vmean_exc -56.69413283803096 -56.69580497509778


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  30779.62449682043
set cost params:  1.0 30779.62449682043 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.124092044646
Gradient descend method:  None
RUN  1 , total integrated cost =  5097.124092044569
RUN  2 , total integrated cost =  5097.124092044556
RUN  3 , total integrated cost =  5097.124092044554
RUN  4 , total integrated cost =  5097.1240920445525
RUN  5 , total integrated cost =  5097.124092044549


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5097.124092044549
Control only changes marginally.
RUN  6 , total integrated cost =  5097.124092044549
Improved over  6  iterations in  2.1234461050480604  seconds by  1.9042545318370685e-12  percent.
Problem in initial value trasfer:  Vmean_exc -67.63039708580827 -67.63675998917762
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5307.728360940537
set cost params:  1.0 5307.728360940537 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13014.976520821689
Gradient descend method:  None
RUN  1 , total integrated cost =  13014.972325938646
RUN  2 , total integrated cost =  13014.972325470899
RUN  3 , total integrated cost =  13014.972325470628
RUN  4 , total integrated cost =  13014.972325470582
RUN  5 , total integrated cost =  13014.972325470562
RUN  6 , total integrated cost =  13014.972325470557


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  13014.972325470557
Control only changes marginally.
RUN  7 , total integrated cost =  13014.972325470557
Improved over  7  iterations in  1.7280691247433424  seconds by  3.223479600933388e-05  percent.
Problem in initial value trasfer:  Vmean_exc -61.596110746597944 -61.625908201382025
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  10721.574057804653
set cost params:  1.0 10721.574057804653 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8231.053886800997
Gradient descend method:  None
RUN  1 , total integrated cost =  8231.053647468047
RUN  2 , total integrated cost =  8231.053645458378
RUN  3 , total integrated cost =  8231.053645369258
RUN  4 , total integrated cost =  8231.053645366208
RUN  5 , total integrated cost =  8231.053645366117
RUN  6 , total integrated cost =  8231.053645366104
RUN  7 , total integrated cost =  8231.053645366103


...

RUN  3 , total integrated cost =  14411.429606760175
Control only changes marginally.
RUN  3 , total integrated cost =  14411.429606760175
Improved over  3  iterations in  1.0132138803601265  seconds by  8.695476417130976e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67653561740965 -56.6765686029087
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  141553.67799756842
set cost params:  1.0 141553.67799756842 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23310.076295403447
Gradient descend method:  None
RUN  1 , total integrated cost =  23310.055649735208
RUN  2 , total integrated cost =  23310.055635561803
RUN  3 , total integrated cost =  23310.05563556179


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  23310.05563556179
Control only changes marginally.
RUN  4 , total integrated cost =  23310.05563556179
Improved over  4  iterations in  1.0660891719162464  seconds by  8.863051925800391e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.70036304854235 -56.70038532838623
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  121407.46911418952
set cost params:  1.0 121407.46911418952 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32974.183292797374
Gradient descend method:  None
RUN  1 , total integrated cost =  32974.1544374076
RUN  2 , total integrated cost =  32974.15443740758


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  32974.15443740758
Control only changes marginally.
RUN  3 , total integrated cost =  32974.15443740758
Improved over  3  iterations in  0.9698947090655565  seconds by  8.750903558052414e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.703673189452005 -56.70366118671825
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.49594088209161624
Gradient descend method:  None
RUN  1 , total integrated cost =  0.16582686411332842
RUN  2 , total integrated cost =  0.1658268641133284
RUN  3 , total integrated cost =  0.16582686411332828
RUN  4 , total integrated cost =  0.16582686411332817


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  0.16582686411332817
Control only changes marginally.
RUN  5 , total integrated cost =  0.16582686411332817
Improved over  5  iterations in  0.5181475132703781  seconds by  66.56317918096241  percent.
Problem in initial value trasfer:  Vmean_exc -68.12663574998521 -68.128415636246
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  43.50465519796567
Gradient descend method:  None
RUN  1 , total integrated cost =  2.6631335678617605
RUN  2 , total integrated cost =  2.6601890771111436
RUN  3 , total integrated cost =  2.660033353950563
RUN  4 , total integrated cost =  2.6594627328199216
RUN  5 , total integrated cost =  2.659423993215301
RUN  6 , total integrated cost =  2.6594239932152934
RUN  7 , total integrated cost =  2.65942399321529


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  2.6594239932152894
RUN  9 , total integrated cost =  2.6594239932152894
Control only changes marginally.
RUN  9 , total integrated cost =  2.6594239932152894
Improved over  9  iterations in  0.8156333491206169  seconds by  93.88703581004442  percent.
Problem in initial value trasfer:  Vmean_exc -67.25288004086576 -67.2614391753404
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13.217794804170099
Gradient descend method:  None
RUN  1 , total integrated cost =  0.7854657268306058
RUN  2 , total integrated cost =  0.78536577917107
RUN  3 , total integrated cost =  0.7853653107599771
RUN  4 , total integrated cost =  0.7853653107599727
RUN  5 , total integrated cost =  0.7853653107599724


...

RUN  110 , total integrated cost =  13141.457854386184
Control only changes marginally.
RUN  111 , total integrated cost =  13141.457854386184
Improved over  111  iterations in  7.133327027782798  seconds by  7.940972797744962  percent.
Problem in initial value trasfer:  Vmean_exc -56.67086839857441 -56.671101114960884
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23087.891247731382
Gradient descend method:  None
RUN  1 , total integrated cost =  22059.874539359094
RUN  2 , total integrated cost =  22029.94060697058
RUN  3 , total integrated cost =  22022.403401760013
RUN  4 , total integrated cost =  22021.048546073755
RUN  5 , total integrated cost =  22020.632803758122
RUN  6 , total integrated cost =  22020.32243380443
RUN  7 , total integrated cost =  22020.196155227597
RUN  8 , total integrated cost =  22019.79586821967
RUN  9 , total integrated cost =  22019.032927739794
RU

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  83 , total integrated cost =  21999.396868727672
Improved over  83  iterations in  5.318708129227161  seconds by  4.714568200812479  percent.
Problem in initial value trasfer:  Vmean_exc -56.698808726513185 -56.69892406063665
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32660.246299119335
Gradient descend method:  None
RUN  1 , total integrated cost =  31650.10470371777
RUN  2 , total integrated cost =  31629.87458874747
RUN  3 , total integrated cost =  31624.3278194404
RUN  4 , total integrated cost =  31623.85122043844
RUN  5 , total integrated cost =  31623.515831163142
RUN  6 , total integrated cost =  31623.21367104424
RUN  7 , total integrated cost =  31623.038057325648
RUN  8 , total integrated cost =  31622.88094134415
RUN  9 , total integrated cost =  31621.612484924855
RUN  10 , total integrated cost =  31620.326535718807
RUN  11 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  148 , total integrated cost =  31606.0497118449
Improved over  148  iterations in  9.67465977370739  seconds by  3.2277668013264957  percent.
Problem in initial value trasfer:  Vmean_exc -56.703934061602666 -56.70392276310143


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.16582686411332817
Gradient descend method:  None
RUN  1 , total integrated cost =  0.16582686411332817
Control only changes marginally.
RUN  1 , total integrated cost =  0.16582686411332817
Improved over  1  iterations in  0.15415598079562187  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -68.12663574998521 -68.128415636246
-------  15 0.450

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.6594239932152894
Control only changes marginally.
RUN  1 , total integrated cost =  2.6594239932152894
Improved over  1  iterations in  0.1634704153984785  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.25288004086576 -67.2614391753404
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.7853653107599722
Gradient descend method:  None
RUN  1 , total integrated cost =  0.7853653107599722
Control only changes marginally.
RUN  1 , total integrated cost =  0.7853653107599722
Improved over  1  iterations in  0.16021306067705154  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -71.25559913876626 -71.27409267689116
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.141824907052

ERROR:root:Problem in initial value trasfer


...

RUN  1 , total integrated cost =  22037.082113565386
Control only changes marginally.
RUN  1 , total integrated cost =  22037.082113565386
Improved over  1  iterations in  0.1808321438729763  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69893297075887 -56.699089366776576
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.2062952665042063
Gradient descend method:  None
RUN  1 , total integrated cost =  0.2062952665042063
Control only changes marginally.
RUN  1 , total integrated cost =  0.2062952665042063
Improved over  1  iterations in  0.1533332858234644  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -76.34126323051855 -76.372979417922
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13141.457854

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13141.457854386184
Control only changes marginally.
RUN  1 , total integrated cost =  13141.457854386184
Improved over  1  iterations in  0.17987903021275997  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67086839857441 -56.671101114960884
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  21999.396868727672
Gradient descend method:  None
RUN  1 , total integrated cost =  21999.396868727672
Control only changes marginally.
RUN  1 , total integrated cost =  21999.396868727672
Improved over  1  iterations in  0.17809118330478668  seconds by  0.0  percent.


ERROR:root:Problem in initial value trasfer


Problem in initial value trasfer:  Vmean_exc -56.698808726513185 -56.69892406063665
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  31606.0497118449
Gradient descend method:  None
RUN  1 , total integrated cost =  31606.0497118449
Control only changes marginally.
RUN  1 , total integrated cost =  31606.0497118449
Improved over  1  iterations in  0.1781440880149603  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.703934061602666 -56.70392276310143
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False]

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
